# Momentum by crypto theme: does it hold within themes, and does liquidity explain it?

**Questions**
1. Does weekly momentum (same design as the base strategy) work inside each theme?
2. Do the themes differ in *measured* liquidity, and does liquidity line up with momentum?
3. Does a portfolio of theme strategies do better or worse than the all-coin strategy?

**Important limits**
* Themes with 2 coins can only rank the two coins, so the strategy is a single pair bet
  (long the recent winner, short the recent loser). That is much noisier than a 7-coin book.
* With 3 themes, a link between liquidity and momentum cannot be established statistically.
  The notebook reports the pattern and the tests, but cannot prove the mechanism.

Themes are defined here before looking at returns, and are not chosen from the results.

In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cryptolib import *  # noqa: F401,F403

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

UNIVERSE = ["BTCUSDT", "ETHUSDT", "ADAUSDT", "BNBUSDT", "XRPUSDT", "DOTUSDT", "MATICUSDT"]
START = "2020-01-01"
END = None
THEMES = {
    "Layer1": ["BTC", "ETH"],
    "DeFi": ["ADA", "DOT"],
    "Tokens": ["BNB", "XRP", "MATIC"],
}
BASE_FORMATION_WEEKS = 2

## 1. Data and common sample

In [ ]:
close_d, qvol_d = load_universe(UNIVERSE, START, END)
px_d = common_window(close_d)          # daily, every coin present
px = to_weekly(px_d)
ret = px.pct_change().dropna()
print(f"Common daily window: {px_d.index[0].date()} -> {px_d.index[-1].date()}")
print(f"Weekly returns: {len(ret)}   coins: {list(ret.columns)}")

## 2. Measured liquidity and volatility by coin
Liquidity is measured as average daily quote-currency (USDT) volume over the common window.
Volatility is annualized daily volatility over the same window. Theme averages are shown
after the coin table.

In [ ]:
win = px_d.index
liq = qvol_d.loc[win[0]:win[-1], px_d.columns].mean() / 1e6
vol_ann = px_d.pct_change().dropna().std() * np.sqrt(PPY_DAILY)
coin_table = pd.DataFrame({"Avg daily USD volume (mn)": liq, "Ann. volatility (daily)": vol_ann})
coin_table["Theme"] = [next((t for t, c in THEMES.items() if coin in c), "none")
                       for coin in coin_table.index]
print(coin_table.sort_values("Avg daily USD volume (mn)", ascending=False).to_string())
print("\nTheme averages:")
print(coin_table.groupby("Theme")[["Avg daily USD volume (mn)", "Ann. volatility (daily)"]]
      .mean().sort_values("Avg daily USD volume (mn)", ascending=False).to_string())

## 3. Momentum within each theme (net of 20 bps)
Each theme uses the same rank-demeaned, normalized, 2-week formation, 1-week holding design
as the base strategy, and ranks only coins inside the theme. The overall strategy (all coins)
is shown for comparison.

In [ ]:
theme_bt = {}
for name, coins in THEMES.items():
    cols = [c for c in coins if c in ret.columns]
    if len(cols) < 2:
        print(f"Skipping {name}: fewer than 2 coins available ({cols})")
        continue
    theme_bt[name] = momentum_backtest(ret[cols], BASE_FORMATION_WEEKS)
    print(f"{name}: {len(cols)} coins -> {cols}")

overall_bt = momentum_backtest(ret, BASE_FORMATION_WEEKS)

summary = metrics_table(
    {**{f"{k} (net)": v["net"] for k, v in theme_bt.items()},
     "Overall, all coins (net)": overall_bt["net"]},
    PPY_WEEKLY)
print()
print(summary.to_string())

## 4. Turnover and costs by theme
Two-coin themes flip their whole position most weeks, so costs take a large share of the
gross return.

In [ ]:
cost_table = pd.DataFrame({k: cost_summary(v, PPY_WEEKLY) for k, v in theme_bt.items()})
cost_table["Overall"] = cost_summary(overall_bt, PPY_WEEKLY)
print(cost_table.to_string())

## 5. Do the themes beat the overall strategy?
A portfolio that averages the theme books (equal weight, each a unit-gross long/short book)
is compared with the all-coin strategy. The test is the HAC t-test on the return difference.
Both are on identical dates.

In [ ]:
theme_net = pd.DataFrame({k: v["net"] for k, v in theme_bt.items()})
combined_net = theme_net.mean(axis=1)

rows = []
for name in theme_net.columns:
    t_vs_zero = mean_test_hac(theme_net[name])
    t_vs_overall = mean_diff_test(theme_net[name], overall_bt["net"])
    rows.append({"Strategy": name,
                 "Net mean (wk)": t_vs_zero["mean"],
                 "p (mean > 0)": t_vs_zero["p-value"],
                 "Mean diff vs overall (wk)": t_vs_overall["mean diff"],
                 "p (diff vs overall)": t_vs_overall["p-value"]})
c_zero = mean_test_hac(combined_net)
c_overall = mean_diff_test(combined_net, overall_bt["net"])
rows.append({"Strategy": "Equal-weight of themes",
             "Net mean (wk)": c_zero["mean"],
             "p (mean > 0)": c_zero["p-value"],
             "Mean diff vs overall (wk)": c_overall["mean diff"],
             "p (diff vs overall)": c_overall["p-value"]})
print(pd.DataFrame(rows).set_index("Strategy").to_string())

print("\nCorrelation of weekly net returns:")
corr_df = pd.concat([theme_net, combined_net.rename("Equal-weight of themes"),
                     overall_bt["net"].rename("Overall")], axis=1)
print(corr_df.corr().to_string())

## 6. Does liquidity line up with momentum?
With three themes this is a descriptive comparison, not a test. Each row is one theme.
A ranking that matches liquidity would be consistent with the hypothesis. A reverse ranking
would argue against it. Neither would establish causation.

In [ ]:
theme_liq = coin_table.groupby("Theme")["Avg daily USD volume (mn)"].mean()
theme_sharpe = summary.loc["Sharpe"].rename(lambda s: s.replace(" (net)", ""))
link = pd.DataFrame({"Avg daily USD volume (mn)": theme_liq,
                     "Net Sharpe": theme_sharpe.reindex(theme_liq.index)})
print(link.sort_values("Avg daily USD volume (mn)", ascending=False).to_string())
print("\nSpearman rank correlation (liquidity vs Sharpe), with n=3 themes:")
rho = link.corr(method="spearman").iloc[0, 1]
print(f"  rho = {rho:.2f}   (n = {len(link)}; cannot be distinguished from chance)")

## 7. Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, v in theme_bt.items():
    (1 + v["net"].fillna(0)).cumprod().plot(ax=axes[0], label=name)
(1 + overall_bt["net"].fillna(0)).cumprod().plot(ax=axes[0], label="Overall", color="black", lw=2)
axes[0].set_yscale("log")
axes[0].set_title("Growth of $1, net of costs")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(theme_liq.index, theme_liq.values, color="#1f77b4", alpha=0.7)
axes[1].set_title("Average daily USD volume by theme (mn)")
axes[1].grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("theme_momentum_and_liquidity.png", dpi=150)
plt.show()

## 8. How to read the results
* **Within-theme momentum:** a theme counts as showing momentum only if its net mean is
  positive with a meaningful p-value, which the table in section 5 reports. Two-coin themes
  are noisy by construction.
* **Diversification:** the combined portfolio is only evidence of diversification if its
  return difference from the overall strategy is not explained by chance, and if its
  correlation with the overall strategy is well below one.
* **Liquidity:** the section 6 link is descriptive. Do not describe it as a finding unless
  the sample is much larger (more coins, or a time-series test within coins).

In [ ]:
print("Summary of this run (all numbers are computed above):")
best = summary.loc["Sharpe"].idxmax()
print(f"  Highest net Sharpe: {best} = {summary.loc['Sharpe'].max():.2f}")
print(f"  Overall net Sharpe: {summary.loc['Sharpe', 'Overall, all coins (net)']:.2f}")
print(f"  Combined-theme net Sharpe: {perf_metrics(combined_net, PPY_WEEKLY)['Sharpe']:.2f}")